# Notebook #6: Single cell Immunosenescence

Here is where the analysis deviates a lot from the original authors. I have a heavy background in senescent states and how they drive disease and a dysfunctional state. I wanted to apply that background to a new pathology.

Therefore, this analysis uses well known senescent and immune dysfunction markers to assess senescent states and dysfunction states in each immune cell in the object.

These states will then be used in downstream analysis of endometrial lesions using spatial transcriptomics.

In [1]:
!pip install seaborn \
matplotlib \
scanpy \
numpy \
anndata \
pandas \
adjustText \
scipy \
statsmodels

In [47]:
# -- Load libraries
from pathlib import Path
import re

import scanpy as sc
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
from adjustText import adjust_text

import anndata as ad
ad.settings.allow_write_nullable_strings = True

In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount = True)

Mounted at /content/drive


In [4]:
# -- Paths
dataset = "GSE179640"

project_dir = Path("/content/drive/MyDrive/endo-immune-atlas")

input_path = (project_dir
              / "data"
              / "interim"
              / dataset
              / "immune_annotated.h5ad"
)


output_path_results = (project_dir
                       / "results"
                       / dataset
                       / "immunosenescence")


output_path_figures = (project_dir
                       / "figures"
                       / dataset
                       / "immunosenescence")

output_path_data.mkdir(parents=True, exist_ok=True)
output_path_results.mkdir(parents=True, exist_ok=True)
output_path_figures.mkdir(parents=True, exist_ok=True)

In [38]:
# -- Color Palettes
tissue_palette = {
    "Ctrl": "#7FA25C",
    "EuE":  "#028090",
    "EcP":  "#456990",
    "EcO":  "#CE7DA5"
}

immune_palette = {
    "Mono-C": "#E07B54",
    "Mono-NC": "#F2A65A",
    "TRM": "#C14F3A",
    "cDC1": "#7B4FA6",
    "cDC2": "#A87DC2",
    "pDC": "#C9A8E0",
    "NK-CD16+": "#2D8C6E",
    "NK-CD16-": "#6DBF9E",
    "CD4 T": "#0B4A7E",
    "CD8 T": "#2E6FA3",
    "Treg": "#5B9EC9",
    "γδ T": "#A8CBE0",
    "B": "#E8C84A",
}


label_palette = {
    "DYS-high only": "#5B9EC9",
    "SEN-high only": "#E07B54",
    "SEN-high & DYS-high": "#C14F3A",
    "Other": "#D3D3D3",
}


# -- Setting Tissue Type Hue Order
hue_order_tissue = ["Ctrl",
                    "EuE",
                    "EcO",
                    "EcP"]
legend_order = [
    "DYS-high only",
    "SEN-high only",
    "SEN-high & DYS-high",
    "Other"
]


In [6]:
# -- Settings (Order, col_names, etc.)
tissue_order = ["Ctrl", "EuE", "EcO", "EcP"]

label_order = [
    "Dysfunction-high only",
    "SEN-high only",
    "SEN-high & Dysfunction-high",
    "Other",
]

cell_type_col = "cell_type"
plot_cell_type_col = "cell_type_short"
tissue_col = "tissue_type"

In [7]:
# -- Import object
immune_obj = sc.read_h5ad(input_path)

In [10]:
# -- Filter out Proliferating group
if "Proliferating" in immune_obj.obs["immune_cell_type"].astype(str).unique():
    raise ValueError(
        "Proliferating cells are still present. Remove them in notebook 05."
    )


immune_obj.obs["immune_cell_type"] = (
    immune_obj.obs["immune_cell_type"]
    .astype("category")
    .cat.remove_unused_categories()
)

immune_obj.obs["cell_type_short"] = (
    immune_obj.obs["cell_type_short"]
    .astype("category")
    .cat.remove_unused_categories()
)

print(immune_obj)
print(immune_obj.obs["immune_cell_type"].value_counts())

AnnData object with n_obs × n_vars = 27882 × 30907
    obs: 'sample_id', 'patient_id', 'tissue_type', 'condition', 'lesion_site', 'dataset', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribos', 'pct_counts_ribos', 'total_counts_hemos', 'pct_counts_hemos', 'n_genes', 'n_counts', 'outlier_mt', 'doublet_score', 'predicted_doublet', 'leiden_res_0.01', 'leiden_res_0.02', 'leiden_res_0.05', 'cluster_label', 'leiden_res_0.20', 'leiden_res_0.50', 'leiden_res_1.00', 'predicted_labels', 'over_clustering', 'majority_voting', 'conf_score', 'immune_cell_type', 'cell_type_short'
    var: 'hemos', 'ribos', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'cluster_label_colors', 'hvg', 'immune_cell_type_colors', 'leiden_res_0.01', 'leiden_res_0.01_colors', 'leiden_res_0.02', 'leiden_res_

In [11]:
#==========================================================================
# SENESCENT MARKERS
#==========================================================================
# - Core/Checkpoint Inbibitors
core_sen_markers = ["CDKN1A",
                    "CDKN2A",
                    "CDKN2B",
                    "TP53",
                    "RB1",
                    "HMGA1"]

# - SASP
sasp_markers = ["IL6",
                "CXCL8",
                "IL1B",
                "TNF",
                "MMP3",
                "MMP9",
                "CXCL1",
                "CXCL2",
                "CXCL10",
                "CCL2",
                "CCL5",
                "VEGFA",
                "IGFBP3",
                "IGFBP7"]

# - Innate-Specific
innate_sen_markers = ["MARCO",
                      "APOE",
                      "SPP1",
                      "CXCR2",
                      "GLB1",
                      "PTPRC",
                      "CD68",
                      "TIGIT",
                      "CD44"]

# - Adaptive-Specific
adaptive_sen_markers = ["B3GAT1",
                        "KLRG1",
                        "EOMES",
                        "TBX21",
                        "LAG3",
                        "PDCD1"]


#==========================================================================
# EXHAUSTION/DYSFUNCTION MARKERS
#==========================================================================
t_cell_exhaustion_markers = ["PDCD1",
                             "HAVCR2",
                             "LAG3",
                             "TIGIT",
                             "TOX",
                             "TOX2",
                             "NR4A1",
                             "CXCL13",
                             "ENTPD1",
                             "KLRG1",
                             "ST2"]

nk_exhaustion_markers = ["TIGIT",
                         "LAG3",
                         "PDCD1",
                         "KIR2DL1",
                         "KIR2DL2",
                         "KIR3DL1",
                         "HAVCR2",
                         "CD96",
                         "KLRC1"]

mac_dysfunction_markers = ["MARCO",
                           "CD274",
                           "SIGLEC1",
                           "MERTK",
                           "IL10",
                           "TGFB1",
                           "VSIG4"]

b_cell_exhaustion_markers = ["PDCD1",
                             "TOX",
                             "FAS",
                             "CD39",
                             "FCRL4",
                             "FCRL5"]

dc_dysfunction = {
    "cDC1": ["IDO1", "CD274", "LAG3", "TIGIT"],
    "cDC2": ["CD274", "IL10", "TIGIT", "FCGR2B"],
    "pDC":  ["CD274", "TIGIT", "LAG3", "LILRB4"]
}

In [13]:
#==========================================================================
# IMMUNE CELL COMPARTMENTS
#==========================================================================

# -- Immune Cell Compartments
innate_cells = [
    "Classical Monocytes",
    "Non-Classical Monocytes",
    "Tissue Resident Macrophages",
    "cDC1",
    "cDC2",
    "pDC",
    "CD16+ NK cells",
    "CD16- NK cells",
]

adaptive_cells = [
    "CD4 T cells",
    "CD8 T cells",
    "Tregs",
    "γδ T cells",
    "B cells"
]

# -- Cell types

cell_type_map = {
    "t_cells": ["CD4 T cells", "CD8 T cells", "Tregs", "γδ T cells"],
    "nk_cells": ["CD16+ NK cells", "CD16- NK cells"],
    "myeloids": ["Classical Monocytes", "Non-Classical Monocytes", "Tissue Resident Macrophages"],
    "b_cells": ["B cells"],
    "DCs": ["cDC1", "cDC2", "pDC"]

}

immune_obj.obs["lineage"] = np.where(
    immune_obj.obs["immune_cell_type"].isin(innate_cells),
    "Innate",
    "Adaptive"
)

In [14]:
# ==========================================================================
# SMALL HELPER FUNCTIONS
# ==========================================================================

def subset_immune(adata, cell_types):
    return adata[
        adata.obs["immune_cell_type"].isin(cell_types)
    ].copy()


def scoring(adata, score_markers, score_name):
    available_markers = [
        marker for marker in score_markers
        if marker in adata.var_names
    ]

    missing_markers = sorted(
        set(score_markers) - set(available_markers)
    )

    if len(available_markers) < 2:
        raise ValueError(
            f"Not enough markers available for {score_name}."
        )

    sc.tl.score_genes(
        adata,
        gene_list=available_markers,
        score_name=score_name,
        random_state=0
    )

    print(f"{score_name}: {len(available_markers)} markers used")

    if missing_markers:
        print(f"Missing markers: {missing_markers}")


def safe_filename(label):
    return re.sub(
        r"[^A-Za-z0-9_-]+",
        "_",
        str(label)
    ).strip("_")

In [15]:
# -- Cell counts per cell type

cell_counts = (
    immune_obj.obs
    .groupby(["tissue_type", "cell_type_short"], observed=True)
    .size()
    .reset_index(name="n_cells")
)

cell_counts_long = (
    cell_counts
    .pivot(
        index="cell_type_short",
        columns="tissue_type",
        values="n_cells"
    )
    .fillna(0)
    .reindex(columns=hue_order_tissue)
)

plt.figure(figsize=(8, 7))
sns.heatmap(
    cell_counts_long,
    annot=True,
    fmt="g",
    cmap="YlGnBu",
    linewidths=0.5
)
plt.title("Cell Counts by Tissue Type")
plt.xlabel("")
plt.ylabel("")
plt.tight_layout()
plt.savefig(
    output_path_figures / "06_cell_counts_per_tissue_cell_type.png",
    bbox_inches="tight",
    dpi=300
)
plt.close()

In [16]:
# ==========================================================================
# SENESCENCE SCORING
# ==========================================================================

scoring(immune_obj, core_sen_markers, "core_sen_score")
scoring(immune_obj, sasp_markers, "sasp_score")

innate_obj = subset_immune(immune_obj, innate_cells)
scoring(innate_obj, innate_sen_markers, "innate_sen_score")

immune_obj.obs["innate_sen_score"] = np.nan
immune_obj.obs.loc[
    innate_obj.obs_names,
    "innate_sen_score"
] = innate_obj.obs["innate_sen_score"]

adaptive_obj = subset_immune(immune_obj, adaptive_cells)
scoring(adaptive_obj, adaptive_sen_markers, "adaptive_sen_score")

immune_obj.obs["adaptive_sen_score"] = np.nan
immune_obj.obs.loc[
    adaptive_obj.obs_names,
    "adaptive_sen_score"
] = adaptive_obj.obs["adaptive_sen_score"]


# ==========================================================================
# COMPOSITE SENESCENCE SCORE
# ==========================================================================

innate_mask = immune_obj.obs["lineage"] == "Innate"
adaptive_mask = immune_obj.obs["lineage"] == "Adaptive"

immune_obj.obs["composite_sen_score"] = np.nan

immune_obj.obs.loc[
    innate_mask,
    "composite_sen_score"
] = (
    immune_obj.obs.loc[
        innate_mask,
        ["core_sen_score", "sasp_score", "innate_sen_score"]
    ]
    .mean(axis=1)
)

immune_obj.obs.loc[
    adaptive_mask,
    "composite_sen_score"
] = (
    immune_obj.obs.loc[
        adaptive_mask,
        ["core_sen_score", "sasp_score", "adaptive_sen_score"]
    ]
    .mean(axis=1)
)

core_sen_score: 6 markers used
sasp_score: 14 markers used
innate_sen_score: 9 markers used
adaptive_sen_score: 6 markers used


In [17]:
# ==========================================================================
# FIGURE 6_1: SENESCENCE SCORES ON UMAP
# ==========================================================================

fig = sc.pl.umap(
    immune_obj,
    color=[
        "core_sen_score",
        "sasp_score",
        "innate_sen_score",
        "adaptive_sen_score",
        "composite_sen_score"
    ],
    ncols=2,
    cmap="RdYlBu_r",
    title=[
        "Core Senescence Score",
        "SASP Score",
        "Innate Senescence Score",
        "Adaptive Senescence Score",
        "Composite Senescence Score"
    ],
    show=False,
    return_fig=True
)

plt.savefig(
    output_path_figures / "06_umap_sen_scores.png",
    bbox_inches="tight",
    dpi=300
)
plt.close()

In [20]:
# ==========================================================================
# FIGURE 6_2: SENESCENCE SCORE HEATMAPS
# ==========================================================================

score_cell_map = {
    "core_sen_score": (innate_cells + adaptive_cells, "Core Senescence"),
    "sasp_score": (innate_cells + adaptive_cells, "SASP"),
    "innate_sen_score": (innate_cells, "Innate Senescence"),
    "adaptive_sen_score": (adaptive_cells, "Adaptive Senescence"),
    "composite_sen_score": (innate_cells + adaptive_cells, "Composite Senescence")
}

cell_type_summary = (
    immune_obj.obs
    .groupby(
        ["immune_cell_type", "cell_type_short", "tissue_type"],
        observed=True
    )[
        [
            "core_sen_score",
            "sasp_score",
            "innate_sen_score",
            "adaptive_sen_score",
            "composite_sen_score"
        ]
    ]
    .mean()
    .reset_index()
)

cell_type_to_short = (
    immune_obj.obs
    .drop_duplicates("immune_cell_type")
    .set_index("immune_cell_type")["cell_type_short"]
    .astype(str)
)

for score_col, (cell_order, title) in score_cell_map.items():
    short_order = cell_type_to_short.loc[cell_order].tolist()

    pivot = (
        cell_type_summary
        .pivot(
            index="cell_type_short",
            columns="tissue_type",
            values=score_col
        )
        .reindex(short_order)
        .reindex(columns=hue_order_tissue)
    )

    fig, ax = plt.subplots(
        figsize=(6, max(4, len(short_order) * 0.5))
    )

    sns.heatmap(
        pivot,
        cmap="RdYlBu_r",
        center=0,
        ax=ax,
        linewidths=0.5,
        cbar_kws={"label": "Mean Score"}
    )

    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.savefig(
        output_path_figures / f"06_heatmap_{score_col}.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()

In [21]:
# ==========================================================================
# EXHAUSTION / DYSFUNCTION SCORING
# ==========================================================================

dysfunction_map = {
    "t_cells": {
        "markers": t_cell_exhaustion_markers,
        "score_name": "tcell_exhaustion_score"
    },
    "nk_cells": {
        "markers": nk_exhaustion_markers,
        "score_name": "nk_dysfunction_score"
    },
    "myeloids": {
        "markers": mac_dysfunction_markers,
        "score_name": "macrophage_dysfunction_score"
    },
    "b_cells": {
        "markers": b_cell_exhaustion_markers,
        "score_name": "bcell_exhaustion_score"
    }
}

immune_obj.obs["dysfunction_score"] = np.nan

for subset_name, params in dysfunction_map.items():
    subset = subset_immune(
        immune_obj,
        cell_type_map[subset_name]
    )

    scoring(
        subset,
        params["markers"],
        params["score_name"]
    )

    immune_obj.obs[params["score_name"]] = np.nan
    immune_obj.obs.loc[
        subset.obs_names,
        params["score_name"]
    ] = subset.obs[params["score_name"]]

    immune_obj.obs.loc[
        subset.obs_names,
        "dysfunction_score"
    ] = subset.obs[params["score_name"]]


dc_map = {
    "cDC1": {
        "markers": dc_dysfunction["cDC1"],
        "score_name": "cdc1_dysfunction_score"
    },
    "cDC2": {
        "markers": dc_dysfunction["cDC2"],
        "score_name": "cdc2_dysfunction_score"
    },
    "pDC": {
        "markers": dc_dysfunction["pDC"],
        "score_name": "pdc_dysfunction_score"
    }
}

for dc_type, params in dc_map.items():
    dc_subset = subset_immune(immune_obj, [dc_type])

    scoring(
        dc_subset,
        params["markers"],
        params["score_name"]
    )

    immune_obj.obs[params["score_name"]] = np.nan
    immune_obj.obs.loc[
        dc_subset.obs_names,
        params["score_name"]
    ] = dc_subset.obs[params["score_name"]]

    immune_obj.obs.loc[
        dc_subset.obs_names,
        "dysfunction_score"
    ] = dc_subset.obs[params["score_name"]]


tcell_exhaustion_score: 10 markers used
Missing markers: ['ST2']
nk_dysfunction_score: 8 markers used
Missing markers: ['KIR2DL2']
macrophage_dysfunction_score: 7 markers used
bcell_exhaustion_score: 5 markers used
Missing markers: ['CD39']
cdc1_dysfunction_score: 4 markers used
cdc2_dysfunction_score: 4 markers used
pdc_dysfunction_score: 4 markers used


In [22]:
# ==========================================================================
# FIGURE 6_3: DYSFUNCTION SCORE HEATMAP
# ==========================================================================
dysfunction_pivot = (
    immune_obj.obs
    .groupby(
        ["cell_type_short", "tissue_type"],
        observed=True
    )["dysfunction_score"]
    .mean()
    .unstack()
    .reindex(list(immune_palette.keys()))
    .reindex(columns=hue_order_tissue)
)

fig, ax = plt.subplots(figsize=(6, 8))
sns.heatmap(
    dysfunction_pivot,
    cmap="RdYlBu_r",
    center=0,
    ax=ax,
    linewidths=0.5,
    cbar_kws={"label": "Mean Dysfunction Score"}
)
ax.set_title("Dysfunction Score per Cell Type across Tissue Types")
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(
    output_path_figures / "06_dysfunction_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)
plt.close()

In [24]:
# ==========================================================================
# SENESCENCE VS DYSFUNCTION CORRELATION
# ==========================================================================

spearman_sen_dys_results = []

for cell_type in immune_obj.obs["immune_cell_type"].cat.categories:
    subset = (
        immune_obj.obs[
            immune_obj.obs["immune_cell_type"] == cell_type
        ]
        .dropna(
            subset=["composite_sen_score", "dysfunction_score"]
        )
    )

    if len(subset) < 30:
        continue

    corr, pval = spearmanr(
        subset["composite_sen_score"],
        subset["dysfunction_score"]
    )

    spearman_sen_dys_results.append(
        {
            "immune_cell_type": cell_type,
            "cell_type_short": subset["cell_type_short"].iloc[0],
            "n_cells": len(subset),
            "spearman_r": corr,
            "pval": pval
        }
    )

sen_vs_dys_spearman_df = pd.DataFrame(spearman_sen_dys_results)

if not sen_vs_dys_spearman_df.empty:
    sen_vs_dys_spearman_df["pval_adj"] = multipletests(
        sen_vs_dys_spearman_df["pval"],
        method="fdr_bh"
    )[1]

    sen_vs_dys_spearman_df["significant"] = (
        sen_vs_dys_spearman_df["pval_adj"] < 0.05
    )

    sen_vs_dys_spearman_df = (
        sen_vs_dys_spearman_df
        .sort_values("spearman_r", ascending=False)
    )

In [25]:
# ==========================================================================
# FIGURE 6_4: SENESCENCE VS DYSFUNCTION
# ==========================================================================

fig, axes = plt.subplots(2, 1, figsize=(14, 12))

sns.scatterplot(
    data=immune_obj.obs.dropna(
        subset=["composite_sen_score", "dysfunction_score"]
    ),
    x="composite_sen_score",
    y="dysfunction_score",
    hue="cell_type_short",
    palette=immune_palette,
    s=20,
    alpha=0.4,
    ax=axes[0]
)
axes[0].axvline(x=0, color="grey", linewidth=0.8, linestyle="--")
axes[0].axhline(y=0, color="grey", linewidth=0.8, linestyle="--")
axes[0].set_xlabel("Composite Senescence Score")
axes[0].set_ylabel("Dysfunction Score")
axes[0].set_title("Senescence vs Dysfunction")
axes[0].legend(
    title="Cell Type",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

if not sen_vs_dys_spearman_df.empty:
    sns.barplot(
        data=sen_vs_dys_spearman_df,
        x="cell_type_short",
        y="spearman_r",
        hue="cell_type_short",
        legend=False,
        palette=immune_palette,
        ax=axes[1]
    )
    axes[1].axhline(y=0, color="black", linewidth=0.8)
    axes[1].set_ylabel("Spearman r")
    axes[1].set_xlabel("")
    axes[1].set_title(
        "Senescence vs Dysfunction Correlation per Cell Type"
    )
    axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig(
    output_path_figures / "06_sen_dysfunction_spearman_corr.png",
    dpi=300,
    bbox_inches="tight"
)
plt.close()

In [27]:
# ==========================================================================
# ASSIGN HIGH AND LOW STATES
# ==========================================================================

# Top 25% is calculated separately within each cell type.
# This avoids baseline differences between immune lineages determining which
# populations are labeled SEN-high or dysfunction-high.

senescence_threshold = (
    immune_obj.obs
    .groupby("immune_cell_type", observed=True)["composite_sen_score"]
    .transform(lambda x: x.quantile(0.75))
)

dysfunction_threshold = (
    immune_obj.obs
    .groupby("immune_cell_type", observed=True)["dysfunction_score"]
    .transform(lambda x: x.quantile(0.75))
)

immune_obj.obs["senescent_state"] = np.where(
    immune_obj.obs["composite_sen_score"] > senescence_threshold,
    "High",
    "Low"
)

immune_obj.obs["dysfunction_state"] = np.where(
    immune_obj.obs["dysfunction_score"] > dysfunction_threshold,
    "High",
    "Low"
)

immune_obj.obs["senescent_state"] = pd.Categorical(
    immune_obj.obs["senescent_state"],
    categories=["Low", "High"],
    ordered=True
)

immune_obj.obs["dysfunction_state"] = pd.Categorical(
    immune_obj.obs["dysfunction_state"],
    categories=["Low", "High"],
    ordered=True
)

state_check = (
    immune_obj.obs
    .groupby("cell_type_short", observed=True)
    .agg(
        prop_sen_high=(
            "senescent_state",
            lambda x: (x == "High").mean()
        ),
        prop_dys_high=(
            "dysfunction_state",
            lambda x: (x == "High").mean()
        )
    )
)

print(state_check)


                 prop_sen_high  prop_dys_high
cell_type_short                              
B                     0.250190       0.250190
CD4 T                 0.250000       0.250000
CD8 T                 0.250000       0.250000
Mono-C                0.250000       0.250000
Mono-NC               0.250000       0.250000
NK-CD16+              0.250215       0.250215
NK-CD16-              0.250000       0.250000
TRM                   0.250000       0.250000
Treg                  0.250000       0.250000
cDC1                  0.248485       0.248485
cDC2                  0.249867       0.249867
pDC                   0.252427       0.252427
γδ T                  0.249946       0.249946


In [30]:
# ==========================================================================
# SENESCENCE AND DYSFUNCTION CO-OCCURRENCE
# ==========================================================================

immune_obj.obs["sen_dysfunction_label"] = "Other"

immune_obj.obs.loc[
    (immune_obj.obs["senescent_state"] == "High")
    & (immune_obj.obs["dysfunction_state"] == "High"),
    "sen_dysfunction_label"
] = "SEN-high & DYS-high"

immune_obj.obs.loc[
    (immune_obj.obs["senescent_state"] == "High")
    & (immune_obj.obs["dysfunction_state"] == "Low"),
    "sen_dysfunction_label"
] = "SEN-high only"

immune_obj.obs.loc[
    (immune_obj.obs["senescent_state"] == "Low")
    & (immune_obj.obs["dysfunction_state"] == "High"),
    "sen_dysfunction_label"
] = "DYS-high only"

immune_obj.obs["sen_dysfunction_label"] = pd.Categorical(
    immune_obj.obs["sen_dysfunction_label"],
    categories=legend_order,
    ordered=True
)

In [33]:
# ==========================================================================
# PROPORTIONAL CALCULATIONS
# ==========================================================================

prop_summary = (
    immune_obj.obs
    .groupby(
        ["tissue_type", "immune_cell_type", "cell_type_short"],
        observed=True
    )
    .agg(
        n_cells=("immune_cell_type", "size"),
        prop_sen_high=(
            "senescent_state",
            lambda x: (x == "High").mean()
        ),
        prop_dys_high=(
            "dysfunction_state",
            lambda x: (x == "High").mean()
        ),
        prop_co_occurrence=(
            "sen_dysfunction_label",
            lambda x: (x == "SEN-high & DYS-high").mean()
        )
    )
    .reset_index()
)

prop_summary["tissue_type"] = pd.Categorical(
    prop_summary["tissue_type"],
    categories=hue_order_tissue,
    ordered=True
)

prop_summary = prop_summary.sort_values(
    ["cell_type_short", "tissue_type"]
)

In [34]:
# ==========================================================================
# FIGURE 6_5: PROPORTION OF CO-OCCURRENCE
# ==========================================================================

plt.figure(figsize=(9, 8))
ax = sns.scatterplot(
    data=prop_summary,
    x="tissue_type",
    y="cell_type_short",
    size="prop_co_occurrence",
    hue="prop_co_occurrence",
    sizes=(20, 400),
    palette="Reds",
    legend="brief"
)
plt.title("Senescence and Dysfunction Co-occurrence")
ax.legend(
    title="Proportion",
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    borderaxespad=0
)
ax.set_ylabel("")
ax.set_xlabel("")
plt.tight_layout()
plt.savefig(
    output_path_figures / "06_proportion_sen_dysfunctional.png",
    dpi=300,
    bbox_inches="tight"
)
plt.close()

In [40]:
# ==========================================================================
# FIGURE 6_6: LABELED UMAP SHOWING CO-OCCURRENCE
# ==========================================================================

centroids = pd.DataFrame(
    immune_obj.obsm["X_umap"],
    index=immune_obj.obs_names,
    columns=["UMAP1", "UMAP2"]
)

centroids["cell_type_short"] = (
    immune_obj.obs["cell_type_short"].astype(str).values
)

centroids = (
    centroids
    .groupby("cell_type_short", observed=True)[["UMAP1", "UMAP2"]]
    .mean()
)

fig = sc.pl.umap(
    immune_obj,
    color="sen_dysfunction_label",
    palette=label_palette,
    title="Senescence and Dysfunction Co-occurrence",
    show=False,
    return_fig=True
)

ax = fig.axes[0]
texts = []

for immune_cell_type, row in centroids.iterrows():
    texts.append(
        ax.text(
            row["UMAP1"],
            row["UMAP2"],
            immune_cell_type,
            fontsize=6,
            ha="center",
            va="center",
            fontweight="bold"
        )
    )

adjust_text(texts, ax=ax)
plt.savefig(
    output_path_figures / "06_umap_sen_dysfunction.png",
    dpi=300,
    bbox_inches="tight"
)
plt.close()

In [41]:
# ==========================================================================
# DEG FUNCTIONS
# ==========================================================================

def run_deg(adata, state_col, key_added, min_cells=10):
    deg_results = {}

    for cell_type in adata.obs["immune_cell_type"].cat.categories:
        subset = subset_immune(adata, [cell_type])
        counts = subset.obs[state_col].value_counts()

        if not {"High", "Low"}.issubset(counts.index):
            continue

        if counts[["High", "Low"]].min() < min_cells:
            print(
                f"Skipping {cell_type}: "
                f"not enough cells for {state_col}"
            )
            continue

        sc.tl.rank_genes_groups(
            subset,
            groupby=state_col,
            groups=["High"],
            reference="Low",
            method="wilcoxon",
            key_added=key_added
        )

        deg_results[cell_type] = sc.get.rank_genes_groups_df(
            subset,
            group="High",
            key=key_added
        )

    return deg_results


def plot_deg(
    deg_results,
    comparison_name,
    pval_threshold=0.05,
    log2fold_threshold=1.5
):
    for cell_type, result in deg_results.items():
        df = result.copy()
        df["-log10_pval"] = -np.log10(df["pvals_adj"] + 1e-300)
        df["significant"] = (
            (df["pvals_adj"] < pval_threshold)
            & (abs(df["logfoldchanges"]) > log2fold_threshold)
        )

        fig, ax = plt.subplots(figsize=(8, 6))

        ax.scatter(
            df.loc[~df["significant"], "logfoldchanges"],
            df.loc[~df["significant"], "-log10_pval"],
            s=3,
            alpha=0.4,
            color="grey"
        )

        ax.scatter(
            df.loc[df["significant"], "logfoldchanges"],
            df.loc[df["significant"], "-log10_pval"],
            s=5,
            alpha=0.7,
            color="#E07B54"
        )

        top = (
            df[df["significant"]]
            .nlargest(10, "-log10_pval")
        )

        texts = [
            ax.text(
                row["logfoldchanges"],
                row["-log10_pval"],
                row["names"],
                fontsize=7
            )
            for _, row in top.iterrows()
        ]

        if texts:
            adjust_text(
                texts,
                ax=ax,
                arrowprops={
                    "arrowstyle": "-",
                    "color": "black",
                    "lw": 0.5
                }
            )

        ax.axvline(
            x=log2fold_threshold,
            color="red",
            linestyle="--",
            linewidth=0.8
        )
        ax.axvline(
            x=-log2fold_threshold,
            color="red",
            linestyle="--",
            linewidth=0.8
        )
        ax.axhline(
            y=-np.log10(pval_threshold),
            color="blue",
            linestyle="--",
            linewidth=0.8
        )
        ax.set_xlabel("Log2 Fold Change")
        ax.set_ylabel("-log10 adjusted p-value")
        ax.set_title(f"{comparison_name}\n{cell_type}")

        filename = (
            f"06_{safe_filename(cell_type)}_"
            f"{safe_filename(comparison_name)}_volcano.png"
        )

        plt.savefig(
            output_path_figures / filename,
            dpi=300,
            bbox_inches="tight"
        )
        plt.close()


def combine_deg_results(deg_results):
    df_list = []

    for cell_type, df in deg_results.items():
        temp_df = df.copy()
        temp_df.insert(0, "cell_type", cell_type)
        df_list.append(temp_df)

    if not df_list:
        return pd.DataFrame()

    return pd.concat(df_list, ignore_index=True)


In [42]:
# ==========================================================================
# DEG ANALYSIS - SENESCENCE
# ==========================================================================

deg_results_sen = run_deg(
    immune_obj,
    state_col="senescent_state",
    key_added="deg_sen"
)

for cell_type, df in deg_results_sen.items():
    print(f"\n{immune_cell_type}")
    print(
        df[df["pvals_adj"] < 0.05]
        .head(10)[["names", "logfoldchanges", "pvals_adj"]]
    )

plot_deg(
    deg_results_sen,
    comparison_name="SEN-high_vs_SEN-low"
)


# ==========================================================================
# DEG ANALYSIS - DYSFUNCTION
# ==========================================================================

deg_results_dysfunction = run_deg(
    immune_obj,
    state_col="dysfunction_state",
    key_added="deg_dysfunction"
)

for cell_type, df in deg_results_dysfunction.items():
    print(f"\n{cell_type}")
    print(
        df[df["pvals_adj"] < 0.05]
        .head(10)[["names", "logfoldchanges", "pvals_adj"]]
    )

plot_deg(
    deg_results_dysfunction,
    comparison_name="DYS-high_vs_DYS-low"
)


γδ T
    names  logfoldchanges     pvals_adj
0     RB1        1.048023  5.513667e-32
1    CD44        0.441863  7.634989e-28
2  CDKN1A        2.165677  3.625400e-22
3    CD68        0.404967  9.387786e-21
4   HMGA1        1.013638  1.481990e-15
5    FTH1        0.282166  4.632128e-15
6   NAMPT        0.888066  4.632128e-15
7    IL1B        2.013083  9.054513e-15
8    JUNB        0.562039  5.320093e-11
9    GLB1        0.681404  6.342401e-11

γδ T
    names  logfoldchanges  pvals_adj
0     RB1        1.087375   0.000007
1  CDKN1A        2.606384   0.000342
2    CD44        0.585546   0.001614
3    TP53        1.073088   0.028483

γδ T
      names  logfoldchanges      pvals_adj
0      SPP1        5.975997  9.312390e-223
1       PKM        1.399214  9.232276e-136
2       VIM        1.404871  1.636637e-131
3  C15orf48        2.644369  1.326366e-123
4     GAPDH        1.224613  1.829157e-118
5      MMP9        4.874345  3.341699e-117
6      ENO1        1.198473  2.798370e-116
7      CD44  

In [48]:
# ==========================================================================
# SAVING RESULTS
# ==========================================================================
senescence_scores = immune_obj.obs[
    [
        "sample_id",
        "patient_id",
        "tissue_type",
        "condition",
        "lesion_site",
        "immune_cell_type",
        "cell_type_short",
        "lineage",
        "core_sen_score",
        "sasp_score",
        "innate_sen_score",
        "adaptive_sen_score",
        "composite_sen_score",
        "dysfunction_score",
        "senescent_state",
        "dysfunction_state",
        "sen_dysfunction_label"
    ]
].copy()

senescence_scores.to_csv(
    output_path_results / "06_senescence_scores.csv"
)

senescence_summary = (
    immune_obj.obs
    .groupby(
        ["immune_cell_type", "cell_type_short", "tissue_type"],
        observed=True
    )
    .agg(
        n_cells=("immune_cell_type", "size"),
        core_sen_score=("core_sen_score", "mean"),
        sasp_score=("sasp_score", "mean"),
        composite_sen_score=("composite_sen_score", "mean"),
        dysfunction_score=("dysfunction_score", "mean"),
        prop_sen_high=(
            "senescent_state",
            lambda x: (x == "High").mean()
        ),
        prop_dys_high=(
            "dysfunction_state",
            lambda x: (x == "High").mean()
        ),
        prop_co_occurrence=(
            "sen_dysfunction_label",
            lambda x: (x == "SEN-high & DYS-high").mean()
        )
    )
    .reset_index()
)

senescence_summary.to_csv(
    output_path_results / "06_senescence_summary.csv",
    index=False
)

sen_vs_dys_spearman_df.to_csv(
    output_path_results / "06_sen_dys_spearman.csv",
    index=False
)

senescence_high_markers = combine_deg_results(deg_results_sen)
senescence_high_markers.to_csv(
    output_path_results / "06_senescence_high_markers.csv",
    index=False
)

dysfunction_high_markers = combine_deg_results(
    deg_results_dysfunction
)
dysfunction_high_markers.to_csv(
    output_path_results / "06_dysfunction_high_markers.csv",
    index=False
)

output_path_data = (project_dir
                    / "data"
                    / "interim"
                    / dataset

                    )


immune_obj.write_h5ad(
    output_path_data / "immunosenescence.h5ad"
)

print("Saved:")
print(" - immunosenescence.h5ad")
print(" - 06_senescence_scores.csv")
print(" - 06_senescence_summary.csv")
print(" - 06_sen_dys_spearman.csv")
print(" - 06_senescence_high_markers.csv")
print(" - 06_dysfunction_high_markers.csv")

Saved:
 - immunosenescence.h5ad
 - 06_senescence_scores.csv
 - 06_senescence_summary.csv
 - 06_sen_dys_spearman.csv
 - 06_senescence_high_markers.csv
 - 06_dysfunction_high_markers.csv
